# Netflix catalogue exploration

This notebook is a narrative around the library in `src/`. It is not the training path. `make train` fits the model. The numbers in `reports/` are produced by the scripts.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import load_config, project_path
from data.load import load_titles
from data.clean import clean_dataframe, listwise_deletion_count
from features.engineer import FeatureEngineer

config = load_config(ROOT / 'config.yaml')
raw = load_titles(project_path(config, 'raw_csv'))
print(raw.shape)
print('columns', list(raw.columns))
print(listwise_deletion_count(raw, config))


## Why rows are kept

Director is missing on roughly 30% of titles (`not given` in this extract). Dropping those rows would throw away a third of the catalogue. Cleaning keeps them and records `director_missing`. Descriptions and cast are absent entirely, so the text block is the title only. That is why the representation comparison has to earn the text block instead of assuming it.


In [ ]:
cleaned = clean_dataframe(raw, config)
engineered = FeatureEngineer(config).fit_transform(raw)
print(engineered['type'].value_counts())
print('duration percentile describe')
print(engineered['duration_normalised'].describe())
print('Movies and series both have a percentile. Neither is given the other type\'s median runtime.')


## What the fitted segments say

If `reports/k_selection.md` and `models/cluster_profiles.json` exist, the next cell prints the decision and the segment names. Re-run `make select-k` and `make train` if they do not.


In [ ]:
import json
decision = ROOT / 'reports' / 'k_selection.json'
profiles = ROOT / 'models' / 'cluster_profiles.json'
if decision.exists():
    payload = json.loads(decision.read_text(encoding='utf-8'))
    print('representation', payload['representation'], 'k', payload['chosen_k'])
    print(payload['reason'])
else:
    print('k selection has not been run')
if profiles.exists():
    for profile in json.loads(profiles.read_text(encoding='utf-8')):
        print(f"{profile['cluster_id']:2d}  {profile['size']:5d}  {profile['name']}")
